<a href="https://colab.research.google.com/github/Mechanics-Mechatronics-and-Robotics/CV-2026/blob/main/Week_09/Hands_on_Face_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1:1 task (verification)

“Do these two images belong to the same person?”

In [ ]:
!pip install facenet-pytorch matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

In [ ]:
# face detector + alignment
mtcnn = MTCNN(image_size=160)

# face embedding network (FaceNet)
model = InceptionResnetV1(pretrained='vggface2').eval()

print("Models loaded")

Upload 3 images

Example:

person1.jpg

person1_2.jpg

person2.jpg

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
def get_embedding(img_path):

    img = Image.open(img_path)

    face = mtcnn(img)

    if face is None:
        raise ValueError("No face detected")

    # visualize aligned face
    # plt.imshow(face.permute(1,2,0))
    # plt.title(img_path)
    # plt.axis("off")
    # plt.show()

    with torch.no_grad():
        emb = model(face.unsqueeze(0))

    return emb.squeeze().numpy()

In [ ]:
embeddings = {}

for img_name in uploaded.keys():
    embeddings[img_name] = get_embedding(img_name)

print("Embeddings computed")

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

In [ ]:
names = list(embeddings.keys())

for i in range(len(names)):
    for j in range(i+1, len(names)):

        s = cosine_similarity(
            embeddings[names[i]],
            embeddings[names[j]]
        )

        print(f"{names[i]} vs {names[j]} → similarity: {s:.3f}")

In [ ]:
threshold = 0.6

def verify(similarity, threshold):
    return "same person" if similarity > threshold else "different"

for i in range(len(names)):
    for j in range(i+1, len(names)):

        s = cosine_similarity(
            embeddings[names[i]],
            embeddings[names[j]]
        )

        decision = verify(s, threshold)

        print(f"{names[i]} vs {names[j]} → {decision} ({s:.3f})")

If we add a database of 10 people and compare a new image against all of them, what task does it become?